# Análisis de filtrado de full text
Compara qué papers pasan o no el `FullTextFilter` (Step 6).

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.paper import Paper
from utils.intermediate_io import STEP4_FILE, STEP5_FILE, STEP6_FILE, load_model_list

In [2]:
papers_step4 = load_model_list(STEP4_FILE, Paper)
papers_step5 = load_model_list(STEP5_FILE, Paper)
papers_step6 = load_model_list(STEP6_FILE, Paper)

print(f"Step4 (raw full text): {len(papers_step4)}")
print(f"Step5 (clean text): {len(papers_step5)}")
print(f"Step6 (passed filter): {len(papers_step6)}")

Step4 (raw full text): 106
Step5 (clean text): 106
Step6 (passed filter): 90


In [3]:
def pid(p: Paper) -> str:
    return (p.doi or p.title).strip().lower()


def to_df(papers, step_name: str):
    return pd.DataFrame(
        [
            {
                "paper_id": pid(p),
                "title": p.title,
                "doi": p.doi,
                "format": p.full_text.format.value if p.full_text else None,
                "chars": len(p.full_text.content)
                if (p.full_text and p.full_text.content)
                else 0,
                "source": p.source,
                "ft_retrieved_by": p.ft_retrieved_by,
                "step": step_name,
            }
            for p in papers
        ]
    )


df4 = to_df(papers_step4, "step4")
df5 = to_df(papers_step5, "step5")
df6 = to_df(papers_step6, "step6")

transitions = [
    ("step4", df4, "step5", df5),
    ("step5", df5, "step6", df6),
]

dropped_parts = []
for from_step, from_df, to_step, to_df_ in transitions:
    to_ids = set(to_df_["paper_id"])
    dropped = from_df[~from_df["paper_id"].isin(to_ids)].copy()
    dropped["from_step"] = from_step
    dropped["to_step"] = to_step
    dropped_parts.append(dropped)

dropped_all = pd.concat(dropped_parts, ignore_index=True)
dropped_all.head()

,paper_id,title,doi,format,chars,source,ft_retrieved_by,step,from_step,to_step
0,10.1172/jci20401,Mechanisms for pituitary tumorigenesis: the pl...,10.1172/jci20401,pdf,1998,openalex,semantic_scholar,step5,step5,step6
1,10.1186/s40001-025-02310-2,Hydrogel-based nanoparticles: revolutionizing ...,10.1186/s40001-025-02310-2,xml,218225,europe_pmc,europe_pmc,step5,step5,step6
2,10.1016/j.advnut.2025.100545,Tea Polyphenol Epigallocatechin Gallate and th...,10.1016/j.advnut.2025.100545,plain,232021,europe_pmc,elsevier,step5,step5,step6
3,10.1016/j.preme.2026.100071,Engineering next-generation in vitro platforms...,10.1016/j.preme.2026.100071,plain,218288,elsevier,elsevier,step5,step5,step6
4,10.1210/er.2014-1003,Inhibin at 90: From Discovery to Clinical Appl...,10.1210/er.2014-1003,html,279646,openalex,unpaywall,step5,step5,step6


In [4]:
summary = (
    dropped_all.groupby(["from_step", "to_step", "format"])
    .agg(
        n=("paper_id", "count"),
        median_chars=("chars", "median"),
    )
    .reset_index()
    .sort_values(["from_step", "n"], ascending=[True, False])
)
summary

,from_step,to_step,format,n,median_chars
2,step5,step6,plain,9,233911.0
1,step5,step6,pdf,3,9627.0
0,step5,step6,html,2,142347.5
3,step5,step6,xml,2,286358.5


In [5]:
cols = [
    "from_step",
    "to_step",
    "paper_id",
    "title",
    "doi",
    "format",
    "chars",
    "source",
    "ft_retrieved_by",
]
dropped_all[cols].sort_values(["from_step", "to_step", "chars"]).head(50)

,from_step,to_step,paper_id,title,doi,format,chars,source,ft_retrieved_by
0,step5,step6,10.1172/jci20401,Mechanisms for pituitary tumorigenesis: the pl...,10.1172/jci20401,pdf,1998,openalex,semantic_scholar
9,step5,step6,10.1016/j.cryobiol.2009.10.167,153. Changes in alginate bead water status and...,10.1016/j.cryobiol.2009.10.167,plain,4690,crossref,elsevier
7,step5,step6,10.1210/jc.2003-031344,The Novel Somatostatin Analog SOM230 Is a Pote...,10.1210/jc.2003-031344,html,5049,openalex,unpaywall
14,step5,step6,10.1016/j.bone.2009.03.435,Rotator cuff repair augmented with stem-cell i...,10.1016/j.bone.2009.03.435,plain,5557,crossref,elsevier
10,step5,step6,10.1023/a:1008843223500,Gel beads composed of collagen reconstituted i...,10.1023/a:1008843223500,pdf,9627,openalex,openalex
1,step5,step6,10.1186/s40001-025-02310-2,Hydrogel-based nanoparticles: revolutionizing ...,10.1186/s40001-025-02310-2,xml,218225,europe_pmc,europe_pmc
3,step5,step6,10.1016/j.preme.2026.100071,Engineering next-generation in vitro platforms...,10.1016/j.preme.2026.100071,plain,218288,elsevier,elsevier
2,step5,step6,10.1016/j.advnut.2025.100545,Tea Polyphenol Epigallocatechin Gallate and th...,10.1016/j.advnut.2025.100545,plain,232021,europe_pmc,elsevier
12,step5,step6,10.1016/j.addr.2023.114959,Recent advances in endocrine organoids for the...,10.1016/j.addr.2023.114959,plain,233911,elsevier,elsevier
6,step5,step6,10.1007/s40883-025-00541-7,The Use of Organoids in Dentistry: An Overview...,10.1007/s40883-025-00541-7,pdf,238017,scopus,openalex


## Notas
- Compara pérdidas entre pasos consecutivos: `step4 -> step5` y `step5 -> step6`.
- `dropped_all` contiene el detalle de papers que NO pasan entre un paso y otro.
- La salida impresa agrupa por transición para facilitar revisión rápida en terminal/notebook.